# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [1]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import random
import torch

In [2]:
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [3]:
DATA_ROOT = './messy_mashup'
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
STEMS = ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

In [4]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    for genre in GENRES:
        genre_path = os.path.join(root_dir, 'genres_stems', genre)
        if not os.path.exists(genre_path):
            continue

        song_dirs = sorted(glob.glob(os.path.join(genre_path, f'{genre}.*')))
        valid_songs = []

        for song_dir in song_dirs:
            song_name = os.path.basename(song_dir)
            is_valid = True

            for stem_file in STEMS:
                fpath = os.path.join(song_dir, stem_file)
                if not os.path.exists(fpath):
                    is_valid = False
                    break
                if os.path.getsize(fpath) < 4 * 1024:
                    is_valid = False
                    break

            if is_valid:
                valid_songs.append(song_name)

        rng.shuffle(valid_songs)
        split_idx = int(len(valid_songs) * val_split)
        val_songs = valid_songs[:split_idx]
        tr_songs = valid_songs[split_idx:]

        def add_to_dict(target_dict, song_list, g=genre):
            for song in song_list:
                for stem_file in STEMS:
                    stem_key = stem_file.replace('.wav', '')
                    filepath = os.path.join(root_dir, 'genres_stems', g, song, stem_file)
                    target_dict[g][stem_key].append(filepath)

        add_to_dict(val_dataset, val_songs)
        add_to_dict(train_dataset, tr_songs)

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

In [5]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    records = []
    for genre, stems in dataset_dict.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                try:
                    y, _ = librosa.load(file_path, sr=sr)
                    total_duration = librosa.get_duration(y=y, sr=sr)
                    intervals = librosa.effects.split(y, top_db=top_db)

                    max_silence = 0.0
                    silence_type = []

                    if len(intervals) == 0:
                        max_silence = total_duration
                        silence_type.append("entire")
                    else:
                        start_silence = intervals[0][0] / sr
                        end_silence = total_duration - (intervals[-1][1] / sr)

                        if start_silence > max_silence:
                            max_silence = start_silence
                        if end_silence > max_silence:
                            max_silence = end_silence

                        if start_silence >= threshold_sec:
                            silence_type.append("start")
                        if end_silence >= threshold_sec:
                            silence_type.append("end")

                        for i in range(len(intervals) - 1):
                            gap = (intervals[i + 1][0] - intervals[i][1]) / sr
                            if gap > max_silence:
                                max_silence = gap
                            if gap >= threshold_sec and "middle" not in silence_type:
                                silence_type.append("middle")

                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 2),
                            "Silence_Location": ", ".join(silence_type),
                            "File_Path": file_path
                        })
                except Exception:
                    continue
    return pd.DataFrame(records)

df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)
df_silence.head()

,Genre,Stem,Duration,Max_Silence_Sec,Silence_Location,File_Path
0,blues,bass,30.01,5.87,middle,./messy_mashup\genres_stems\blues\blues.00021\...
1,blues,bass,30.01,21.80,start,./messy_mashup\genres_stems\blues\blues.00032\...
2,blues,bass,30.01,17.11,start,./messy_mashup\genres_stems\blues\blues.00096\...
3,blues,bass,30.01,23.68,start,./messy_mashup\genres_stems\blues\blues.00036\...
4,blues,bass,30.01,7.31,middle,./messy_mashup\genres_stems\blues\blues.00039\...


In [6]:
genre_dir = os.path.join(DATA_ROOT, 'genres_stems', GENRE_TO_TEST)
song_folders = sorted(glob.glob(os.path.join(genre_dir, f'{GENRE_TO_TEST}.*')))
target_song_path = song_folders[SONG_INDEX]

stems_audio = []
for stem_file in STEMS:
    stem_path = os.path.join(target_song_path, stem_file)
    y, _ = librosa.load(stem_path, sr=SR, duration=DURATION)
    stems_audio.append(y)

len(stems_audio)

4

In [7]:
stems_stack = np.array(stems_audio)

mix_raw = np.sum(stems_stack, axis=0)

rms_val = np.sqrt(np.mean(mix_raw**2))

max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."

In [8]:
all_stem_sizes = []
corrupted_count = 0
for genre in GENRES:
    genre_path = os.path.join(DATA_ROOT, 'genres_stems', genre)
    if not os.path.exists(genre_path):
        continue
    song_dirs = sorted(glob.glob(os.path.join(genre_path, f'{genre}.*')))
    for song_dir in song_dirs:
        for stem_file in STEMS:
            file_path = os.path.join(song_dir, stem_file)
            if os.path.exists(file_path):
                file_size = os.path.getsize(file_path)
                all_stem_sizes.append(file_size)
                if file_size < 4 * 1024:
                    corrupted_count += 1

all_stem_sizes = np.array(all_stem_sizes)
threshold_low = 5.0491 * 1024 * 1024
threshold_high = 5.0493 * 1024 * 1024
below_low = int(np.sum(all_stem_sizes < threshold_low))
above_high = int(np.sum(all_stem_sizes > threshold_high))

tr_reggae_drums = len(tr['reggae']['drums'])
val_country_vocals = len(val['country']['vocals'])

answers_1_3 = {
    "Q1": int(corrupted_count + below_low),
    "Q2": int(abs(above_high - below_low)),
    "Q3": int(abs(tr_reggae_drums - val_country_vocals))
}
answers_1_3

{'Q1': 1256, 'Q2': 1072, 'Q3': 66}

In [9]:
vocals_df = df_silence[df_silence['Stem'] == 'vocals']
jazz_drums = df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]
jazz_drums_middle_only = jazz_drums[jazz_drums['Silence_Location'] == 'middle']
jazz_drums_10 = jazz_drums[jazz_drums['Max_Silence_Sec'] >= 10]

answers_4_9 = {
    "Q4": int(len(df_silence)),
    "Q5": int(len(vocals_df)),
    "Q6": float(vocals_df['Max_Silence_Sec'].mean()) if len(vocals_df) > 0 else 0.0,
    "Q7": int(len(jazz_drums)),
    "Q8": int(len(jazz_drums_middle_only)),
    "Q9": int(len(jazz_drums_10))
}
answers_4_9

{'Q4': 688, 'Q5': 312, 'Q6': 12.544326923076923, 'Q7': 22, 'Q8': 12, 'Q9': 8}

In [10]:
answers_10_12 = {
    "Q10": int(len(mix_raw)),
    "Q11": float(rms_val),
    "Q12": float(max_val)
}
answers_10_12

{'Q10': 110250, 'Q11': 0.2006196826696396, 'Q12': 0.9600698351860046}